<a href="https://colab.research.google.com/github/pedroManuelP/C-digos-em-Python/blob/main/Lista3_2_SistControle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from numpy.polynomial import Polynomial
import re

def FT_to_EE(g_num, g_den, s):
  n = g_den.degree()
  alpha = g_den.coef[0:-1]
  beta = g_num.coef
  if(s == '1'):
    A = np.identity(n); A = np.roll(A, -1, axis = 0); A[-1][0] = 0; A[-1,:] = -alpha
    B = np.zeros((n, 1)); B[-1, 0] = 1
    C = np.zeros((1, n)); C[0,:] = beta
  if(s == '2'):
    A = np.identity(n); A = np.roll(A, -1, axis = 1); A[0][-1] = 0; A[:, -1] = -alpha
    B = np.zeros((n,1)); B[:,0] = beta.T
    C = np.zeros((1,n)); C[0, -1] = 1
  return A, B, C

def matrizReferencia(A, B, C):
  Ar = np.zeros((1+A.shape[0], 1+C.shape[1]))
  Ar[0, 1:C.shape[1]+1] = C; Ar[1:A.shape[0]+1, 1:A.shape[1]+1] = A

  Br = np.zeros((B.shape[0]+1, 1)); Br[1:B.shape[0]+1, 0] = B[:, 0]
  return Ar, Br

def matrizExpandida(A, B, C, Kr):
  # Matriz expandida para um seguidor de referência do tipo degrau unitário
  Ae = np.zeros((A.shape[0]+C.shape[0], C.shape[1]+1))
  Ae[0:A.shape[0], 0:B.shape[0]] = A+B*Kr[0, 1:Kr.shape[1]]
  Ae[0:B.shape[0], -1] = (B*Kr[0][0])[:,0]; Ae[-1, 0:C.shape[1]] = C

  Be = np.zeros((Ae.shape[0], 1)); Be[-1, 0] = -1
  Ce = np.zeros((1, Ae.shape[1])); Ce[0, 0:C.shape[1]] = C
  return Ae, Be, Ce

def matrizControlabilidade(A, B):
  n = A.shape[0]
  U = np.zeros((n,n))
  for i in range(n):
    U[:,i:i+1] = (np.linalg.matrix_power(A,i))@B
  posto = np.linalg.matrix_rank(U)
  if(posto == A.shape[0]):
    return U, True
  else:
    return U, False

def realimentaçãoDeEstado(A, U, s):
  theta = Polynomial.fromroots(s)
  print('\n'+str(theta))
  coeficientes = (theta.coef).real

  n = A.shape[0]; qc = np.zeros((n,n))
  for i in range(n+1):
    qc += coeficientes[i]*(np.linalg.matrix_power(A,i))
  invU = np.linalg.inv(U)
  linha = np.zeros((1,n)); linha[0][-1] = 1
  return -linha@(invU@qc)

def input_matriz(nome):
    print(f"\nDigite a matriz {nome} (separando elementos por espaços, linhas por ponto e vírgula):")
    print(f"Exemplo: 0 1; -2 -3")
    s = input().strip()
    linhas = s.split(';')
    matriz = []
    for linha in linhas:
        elementos = [float(x) for x in linha.strip().split()]
        matriz.append(elementos)
    return np.array(matriz)


In [3]:
# Escolha do método de entrada
print("="*50)
print("Como deseja definir o sistema?")
print("1 - Através da função de transferência G(s)")
print("2 - Diretamente através das matrizes A, B, C")
opcao = input("Digite 1 ou 2: ").strip()

if opcao == '1':
    # Função G(s)
    print("\nCoeficientes do numerador de G(s): ", end = ''); s = input()
    coef_arr = np.array(re.findall(r'-?\d*\.?\d+', s), dtype=float)
    g_num = Polynomial(np.flip(coef_arr))
    print("Coeficientes do denominador de G(s): ", end = ''); s = input()
    coef_arr = np.array(re.findall(r'-?\d*\.?\d+', s), dtype=float)
    g_den = Polynomial(np.flip(coef_arr))

    # Impressão das funções
    print("G(x) = (" + str(g_num) + ")/(" + str(g_den) + ")\n")

    # Representação dos Espaços de Estados
    print("Como desejar representar a função no espaço de estados?")
    print("Digite 1 para forma canônica controlável;"); print("Digite 2 para forma canônica observável.")
    s = input(); A, B, C = FT_to_EE(g_num, g_den, s)

elif opcao == '2':
    print("\n--- Entrada direta das matrizes de espaço de estados ---")

    # Entrada da matriz A
    A = input_matriz("A")

    # Entrada da matriz B
    print(f"\nMatriz A tem dimensão {A.shape[0]}x{A.shape[1]}")
    print(f"A matriz B deve ter {A.shape[0]} linhas")
    B = input_matriz("B")

    # Verificar dimensão de B
    if B.shape[0] != A.shape[0]:
        print(f"Erro: B deve ter {A.shape[0]} linhas. Reinicie o programa.")
        exit()

    # Entrada da matriz C
    print(f"\nA matriz C deve ter {A.shape[1]} colunas")
    C = input_matriz("C")

    # Verificar dimensão de C
    if C.shape[1] != A.shape[1]:
        print(f"Erro: C deve ter {A.shape[1]} colunas. Reinicie o programa.")
        exit()

    # Se C for um vetor linha, garantir o formato correto
    if len(C.shape) == 1:
        C = C.reshape(1, -1)

    print("\nSistema definido pelas matrizes:")

else:
    print("Opção inválida! Reinicie o programa.")
    exit()

# Impressão das matrizes do sistema
print("\nA = " + str(A))
print("B = " + str(B))
print("C = " + str(C))
# Cálculo das matrizes do estado interno
Ar, Br = matrizReferencia(A, B, C)
print("\nAr = " + str(Ar)); print("Br = " + str(Br))

# Cálculo da matriz de controlabilidade
Ur, controlabilidade = matrizControlabilidade(Ar, Br)
print("Ur = "+str(Ur))
if(controlabilidade):
  print("\nA função é controlável, logo é possível fazer um seguidor de referência.")

  # Cálculo da realimentação de estado para o modelo interno
  print("Informe agora os polos:"); s = input(); p = np.zeros((1,0))
  while(s != ''):
    z = np.array(re.findall(r'-?\d*\.?\d+', s), dtype=float);
    if(z.shape[0] == 2):
      p = np.append(p, complex(z[0],z[1]))
    else:
      p = np.append(p, complex(z[0],0))
    s = input();
  print("s = " + str(p))
  Kr = realimentaçãoDeEstado(Ar, Ur, p)
  print("\nKr = " + str(Kr))

  # Cálculo da matriz expandida para um seguidor de referência do tipo degrau unitário
  Ae, Be, Ce = matrizExpandida(A, B, C, Kr)
  print("\nAe = " + str(Ae)); print("Be = " + str(Be)); print("Ce = " + str(Ce))
else:
  print("A função não é controlável, logo não é possível fazer um seguidor de referência.")



Como deseja definir o sistema?
1 - Através da função de transferência G(s)
2 - Diretamente através das matrizes A, B, C
Digite 1 ou 2: 2

--- Entrada direta das matrizes de espaço de estados ---

Digite a matriz A (separando elementos por espaços, linhas por ponto e vírgula):
Exemplo: 0 1; -2 -3
1 0; 0 1

Matriz A tem dimensão 2x2
A matriz B deve ter 2 linhas

Digite a matriz B (separando elementos por espaços, linhas por ponto e vírgula):
Exemplo: 0 1; -2 -3
1; 0

A matriz C deve ter 2 colunas

Digite a matriz C (separando elementos por espaços, linhas por ponto e vírgula):
Exemplo: 0 1; -2 -3
1 2

Sistema definido pelas matrizes:

A = [[1. 0.]
 [0. 1.]]
B = [[1.]
 [0.]]
C = [[1. 2.]]

Ar = [[0. 1. 2.]
 [0. 1. 0.]
 [0. 0. 1.]]
Br = [[0.]
 [1.]
 [0.]]
Ur = [[0. 1. 1.]
 [1. 1. 1.]
 [0. 0. 0.]]
A função não é controlável, logo não é possível fazer um seguidor de referência.
